# Exploratory analysis: `xc_metadata_unified.csv`

Quick answers for ~18k Xeno-canto rows: species counts, bird vs non-bird (heuristic), vocalization labels, and missing fields.

**Taxon note:** There is no `class` column in the CSV. Non-bird groups below use explicit species codes from this dataset plus simple suffix rules (`_frog`, `_bat`, …) so new rows get a best-effort label. Treat `identity_unknown` / `soundscape` separately.

In [14]:
from pathlib import Path

import pandas as pd

CSV_PATH = Path("xc_metadata_unified.csv")
if not CSV_PATH.is_file():
    CSV_PATH = Path("scripts/xc_metadata_unified.csv")

df = pd.read_csv(CSV_PATH)
df.shape

(17787, 7)

## 1. Dataset shape and preview

In [15]:
df.head(10)

,filepath,species_code,common_name,vocalization_type,quality_rating,duration,source
0,audio/xc/1060250.mp3,north_american_red_squirrel,North American Red Squirrel,call,5,0:07,xeno-canto
1,audio/xc/1020851.mp3,coyote,Coyote,call,5,0:33,xeno-canto
2,audio/xc/1020611.mp3,spring_peeper,Spring Peeper,"call, chorus",5,2:35,xeno-canto
3,audio/xc/1020608.mp3,spring_peeper,Spring Peeper,call,5,2:39,xeno-canto
4,audio/xc/1020607.mp3,spring_peeper,Spring Peeper,call,5,2:26,xeno-canto
5,audio/xc/1017885.mp3,north_american_red_squirrel,North American Red Squirrel,call,5,0:47,xeno-canto
6,audio/xc/1017877.mp3,north_american_red_squirrel,North American Red Squirrel,call,5,1:17,xeno-canto
7,audio/xc/1017876.mp3,north_american_red_squirrel,North American Red Squirrel,"call, rattle",5,0:13,xeno-canto
8,audio/xc/1017859.mp3,north_american_red_squirrel,North American Red Squirrel,call,5,0:14,xeno-canto
9,audio/xc/1014820.mp3,bronze_frog,Bronze Frog,advertisement call,5,1:04,xeno-canto


In [16]:
df.dtypes

filepath               str
species_code           str
common_name            str
vocalization_type      str
quality_rating       int64
duration               str
source                 str
dtype: object

## 2. How many unique species?

Use `species_code` as the stable ID; `common_name` should line up (check mismatches if any).

In [17]:
n_rows = len(df)
n_code = df["species_code"].nunique(dropna=True)
n_name = df["common_name"].nunique(dropna=True)
print(f"Rows: {n_rows:,}")
print(f"Unique species_code: {n_code:,}")
print(f"Unique common_name: {n_name:,}")

Rows: 17,787
Unique species_code: 436
Unique common_name: 436


## 3. Missing / bad data

In [18]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "pct": missing_pct})[missing > 0]

,missing_count,pct
species_code,4,0.02
common_name,4,0.02
vocalization_type,144,0.81


In [19]:
bad = df[df["species_code"].isna() | df["common_name"].isna()]
print(f"Rows with missing species_code or common_name: {len(bad)}")
bad

Rows with missing species_code or common_name: 4


,filepath,species_code,common_name,vocalization_type,quality_rating,duration,source
34,audio/xc/1031093.mp3,NaN,NaN,calling song,4,0:23,xeno-canto
36,audio/xc/1025148.mp3,NaN,NaN,calling song,4,0:20,xeno-canto
38,audio/xc/1024376.mp3,NaN,NaN,calling song,4,1:52,xeno-canto
17173,audio/xc/932093.mp3,NaN,NaN,flight song,3,0:08,xeno-canto


## 4. Bird vs frog vs mammal (and other)

Heuristic `taxon_group` — see docstring. Inspect edge cases with `df.loc[df["taxon_group"] == "bird", "species_code"].unique()` filtered by keywords if you tighten rules later.

In [20]:
OTHER_CODES = frozenset({"identity_unknown", "soundscape"})

AMPHIBIAN_CODES = frozenset(
    {
        "american_bullfrog",
        "american_toad",
        "boreal_chorus_frog",
        "bronze_frog",
        "eastern_gray_treefrog",
        "spring_peeper",
        "wood_frog",
    }
)

MAMMAL_CODES = frozenset(
    {
        "big_brown_bat",
        "canadian_lynx",
        "coyote",
        "collared_pika",
        "eastern_chipmunk",
        "eastern_gray_squirrel",
        "eastern_red_bat",
        "eastern_small-footed_myotis",
        "little_brown_myotis",
        "north_american_porcupine",
        "north_american_red_squirrel",
        "northern_hoary_bat",
        "northern_myotis",
        "silver-haired_bat",
        "singing_vole",
        "taiga_vole",
        "tricolored_bat",
        "white-tailed_deer",
    }
)

INSECT_CODES = frozenset(
    {
        "broad-winged_bush-katydid",
        "marsh_ground_cricket",
        "riley's_tree_cricket",
        "steindachner's_shieldback",
    }
)


def taxon_group(species_code: object) -> str:
    if pd.isna(species_code):
        return "missing"
    c = str(species_code).strip()
    if c in OTHER_CODES:
        return "other_non_species"
    if c in INSECT_CODES:
        return "insect"
    if c in AMPHIBIAN_CODES:
        return "amphibian"
    if c in MAMMAL_CODES:
        return "mammal"
    if c.endswith("_frog") or c.endswith("_toad") or "peeper" in c or "treefrog" in c or "chorus_frog" in c:
        return "amphibian"
    if any(
        x in c
        for x in (
            "_bat",
            "myotis",
            "squirrel",
            "coyote",
            "porcupine",
            "chipmunk",
            "_lynx",
            "_deer",
            "_vole",
            "_pika",
        )
    ):
        return "mammal"
    if "cricket" in c or "katydid" in c or "shieldback" in c:
        return "insect"
    return "bird"


df["taxon_group"] = df["species_code"].map(taxon_group)
counts = df["taxon_group"].value_counts()
counts

taxon_group
bird                 17100
other_non_species      586
mammal                  58
amphibian               34
insect                   5
missing                  4
Name: count, dtype: int64

In [21]:
summary = pd.DataFrame(
    {
        "rows": counts,
        "pct": (counts / len(df) * 100).round(2),
    }
)
summary

,rows,pct
taxon_group,,
bird,17100,96.14
other_non_species,586,3.29
mammal,58,0.33
amphibian,34,0.19
insect,5,0.03
missing,4,0.02


In [22]:
bird_rows = (df["taxon_group"] == "bird").sum()
non_bird = len(df) - bird_rows
print(f"Rows labeled bird (heuristic): {bird_rows:,} ({bird_rows/len(df)*100:.2f}%)")
print(f"Rows not bird: {non_bird:,}")

Rows labeled bird (heuristic): 17,100 (96.14%)
Rows not bird: 687


## 5. Vocalization types

Raw field can combine several tags (e.g. `"call, chorus"`). Below: **raw** value counts, then **split** tags (comma-separated) for a cleaner frequency table.

In [23]:
raw_vc = df["vocalization_type"].value_counts(dropna=False)
print(f"Distinct raw vocalization_type strings: {raw_vc.size:,}")
raw_vc.head(25)

Distinct raw vocalization_type strings: 1,228


vocalization_type
song                                  6564
call                                  3942
flight call                           1632
call, song                             916
nocturnal flight call                  788
uncertain                              270
call, flight call                      236
alarm call                             201
NaN                                    144
alarm call, call                       109
call, chip                              98
drumming                                94
call, contact calls                     94
call, fledgling calls                   58
flight call, song                       58
begging call                            58
call, flight call, song                 46
call, flight call, chip                 44
begging call, call                      38
flight call, song, display              28
flight call, nocturnal flight call      27
alarm call, song                        26
call, drumming                      

In [24]:
def split_voc_tags(x):
    if pd.isna(x):
        return []
    parts = str(x).split(",")
    return [p.strip() for p in parts if p.strip()]


tags = df["vocalization_type"].map(split_voc_tags).explode()
tag_counts = tags.value_counts(dropna=True)
print(f"Distinct tags after splitting commas: {len(tag_counts):,}")
tag_counts.head(40)

Distinct tags after splitting commas: 899


vocalization_type
song                     8191
call                     7046
flight call              2604
nocturnal flight call     868
alarm call                809
uncertain                 274
chip                      176
drumming                  144
begging call              133
contact calls              99
fledgling calls            71
interaction calls          62
subsong                    62
various calls              49
dawn song                  44
duet                       33
whinny                     32
display                    32
calls                      29
flight song                29
rattle                     27
various                    25
pink                       24
winnowing                  23
tink                       23
calls in flight            22
echolocation               21
aberrant                   20
chatter                    19
taking off                 17
whine                      17
jumbled song               17
simple call           

## 6. Optional: rows per species (top and tail)

Useful to spot over-represented taxa or sparse ones.

In [25]:
per_sp = df.groupby("species_code", dropna=False).size().sort_values(ascending=False)
per_sp.head(15)

species_code
identity_unknown           478
white-throated_sparrow     392
song_sparrow               325
red-winged_blackbird       308
swainson's_thrush          297
american_redstart          262
magnolia_warbler           235
american_robin             230
myrtle_warbler             223
american_yellow_warbler    217
red_crossbill              217
ruby-crowned_kinglet       205
purple_finch               190
common_yellowthroat        187
northern_waterthrush       180
dtype: int64

In [26]:
per_sp.tail(15)

species_code
burrowing_owl                   1
calliope_hummingbird            1
canadian_lynx                   1
chuck-will's-widow              1
chinese_blackbird               1
citrine_wagtail                 1
black_guillemot                 1
black-bellied_whistling_duck    1
bell's_vireo                    1
band-tailed_pigeon              1
arctic_redpoll                  1
white-tailed_deer               1
white-throated_swift            1
white-winged_scoter             1
williamson's_sapsucker          1
dtype: int64